# Dataset Load

In [1]:
from sklearn.datasets import load_breast_cancer
dataset = load_breast_cancer()

In [2]:
print(dataset.DESCR)

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [3]:
x, y = dataset.data, dataset.target

# Train Test Split

In [4]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test =  train_test_split(x, y, test_size = 0.2, random_state = 2024)
x_train, x_val, y_train, y_val =  train_test_split(x_train, y_train, test_size = 0.2, random_state = 2024)

In [5]:
print(x_train.shape, x_test.shape, x_val.shape)

(364, 30) (114, 30) (91, 30)


# PyTorch Modeling for Classification Task

## 1.Library Import



In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, Dataset, DataLoader

## 2.Dataset / Data Loader 생성/선언

In [7]:
x_train = torch.Tensor(x_train)
x_test = torch.Tensor(x_test)
x_val = torch.Tensor(x_val)
y_train = torch.Tensor(y_train)
y_test = torch.Tensor(y_test)
y_val = torch.Tensor(y_val)


batch_size = 64
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)


train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_dataloader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

## 3.Model Class 생성 & Model 선언
- Functional Style
    - Functional Type Coding은 각 layer를 함수형태로 구성하고, 이전 layer들 출력값을 다음 단계의 layer 함수의 입력값에 연결시켜주는 형태로 Model의 Layer를 쌓아나가는 방식입니다.

In [8]:
class BinaryClassificationModel(nn.Module):
    def __init__(self):
        super(BinaryClassificationModel, self).__init__()

        self.linear_1 = nn.Linear(x_train.shape[1], 64)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(64,32)
        self.linear_3 = nn.Linear(32,32)
        self.linear_4 = nn.Linear(32,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.linear_1(x)
        x = self.relu(x)
        x = self.linear_2(x)
        x = self.relu(x)
        x = self.linear_3(x)
        x = self.relu(x)
        x = self.linear_4(x)
        x = self.sigmoid(x)

        return x


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BinaryClassificationModel().to(device)

In [10]:
print(model)

BinaryClassificationModel(
  (linear_1): Linear(in_features=30, out_features=64, bias=True)
  (relu): ReLU()
  (linear_2): Linear(in_features=64, out_features=32, bias=True)
  (linear_3): Linear(in_features=32, out_features=32, bias=True)
  (linear_4): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 4.Optimizer, Loss 선언
- Binary Classification Model은 loss function을 BCELoss를 사용합니다.

In [11]:
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [12]:
loss_fn = nn.BCELoss().to(device)

## 5.Model Training
- PyTorch는 Training을 할때, 각 Epoch마다 Batch의 개수만큼의 Iteration을 수행합니다.
- 한 Epoch의 평균 Accuracy는 각 Batch의 Accuracy를 더한 값을 Batch의 개수로 나누어주면 구할 수 있습니다.

In [13]:
from sklearn.metrics import accuracy_score

In [14]:
epochs = 100

for epoch in range(epochs):
  train_cost = 0
  train_accuracy = 0
  train_batch_length = len(train_dataloader)

  model.train()
  for train_inputs, train_labels in train_dataloader:

    ## train mode
    train_inputs = train_inputs.to(device)
    train_labels = train_labels.to(device)

    ## Model 예측
    train_preds = model(train_inputs)

    ## loss 계산
    train_loss = loss_fn(train_preds.flatten(), train_labels)

    ## optimizer 초기화
    optimizer.zero_grad()

    ## Backpropagation
    train_loss.backward()

    ## weight를 next step으로 이동
    optimizer.step()

    train_cost += train_loss.item()
    train_preds = train_preds.flatten().detach().cpu().numpy().round()
    train_labels = train_labels.detach().cpu().numpy()

    train_accuracy += accuracy_score(train_labels, train_preds)


  train_cost /=  train_batch_length
  train_accuracy /= train_batch_length

  ## validation set 평가

  val_cost = 0
  val_accuracy = 0
  val_batch_length = len(val_dataloader)

  model.eval() ## layer들을 inference 모드로 동작하도록 설정 변경
  with torch.no_grad(): ## optimizer가 gradient를 계산하는 연산을 수행하지 않고 inference목적으로만 사용하도록 하여 속도를 높임
    for val_inputs, val_labels in val_dataloader:

      val_inputs = val_inputs.to(device)
      val_labels = val_labels.to(device)

      val_preds = model(val_inputs)
      val_loss = loss_fn(val_preds.flatten(), val_labels)
      val_cost += val_loss.item()


      val_preds = val_preds.flatten().detach().cpu().numpy().round()
      val_labels = val_labels.detach().cpu().numpy()

      val_accuracy += accuracy_score(val_labels, val_preds)

  val_cost /= val_batch_length
  val_accuracy /= val_batch_length

  print("{} / {} epochs => train loss : {:.4f}, train acc : {:.4f}, val loss : {:.4f}, val acc : {:.4f}".format(epoch+1, epochs, train_cost, train_accuracy, val_cost,  val_accuracy))

1 / 100 epochs => train loss : 4.1016, train acc : 0.4972, val loss : 1.4499, val acc : 0.3492
2 / 100 epochs => train loss : 0.9159, train acc : 0.4799, val loss : 0.7425, val acc : 0.7190
3 / 100 epochs => train loss : 0.6124, train acc : 0.7625, val loss : 0.5609, val acc : 0.6751
4 / 100 epochs => train loss : 0.5376, train acc : 0.5956, val loss : 0.4509, val acc : 0.9005
5 / 100 epochs => train loss : 0.4169, train acc : 0.8575, val loss : 0.4132, val acc : 0.8585
6 / 100 epochs => train loss : 0.3763, train acc : 0.8793, val loss : 0.3914, val acc : 0.8663
7 / 100 epochs => train loss : 0.3359, train acc : 0.8913, val loss : 0.3707, val acc : 0.8927
8 / 100 epochs => train loss : 0.3007, train acc : 0.9223, val loss : 0.3567, val acc : 0.8927
9 / 100 epochs => train loss : 0.2831, train acc : 0.9186, val loss : 0.3382, val acc : 0.8819
10 / 100 epochs => train loss : 0.2684, train acc : 0.9212, val loss : 0.3308, val acc : 0.8927
11 / 100 epochs => train loss : 0.2734, train acc

## 6.Model Evaluation

In [15]:
model.eval()
with torch.no_grad():
    y_test_pred = model(x_test.to(device))

In [16]:
y_test_pred = y_test_pred.round().detach().cpu().numpy()
y_test = y_test.detach().cpu().numpy()

In [17]:
from sklearn.metrics import accuracy_score, f1_score

In [18]:
accuracy_score(y_test, y_test_pred)

0.9122807017543859

In [19]:
f1_score(y_test, y_test_pred)

0.9285714285714286